<a href="https://colab.research.google.com/github/chedy322/LLMv1/blob/main/Llmv1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Perform Sementic Search model

# Install langchain




In [ ]:

# laod the data
data= """The Magic Library is located in the clouds. To enter the library, you must whisper the secret password 'Lumos'. Once inside, you can find books that talk.
 Some books are friendly, while others are very grumpy. Do not wake the grumpy books up."""

In [ ]:
!pip install -qU langchain-text-splitters

In [ ]:
# split the text into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=200,
    chunk_overlap=20,
)
chunks=splitter.split_text(data)
chunks

In [ ]:
# Transform the chunks of data into embdeddings
from sentence_transformers import SentenceTransformer
model=SentenceTransformer("all-MiniLM-L6-v2")

# This model has 256 tokens per context window


embeddings=model.encode(chunks,batch_size=32,show_progress_bar=True)
embeddings


In [ ]:
#Install weviate
!pip install -U weaviate-client

In [ ]:
# store the embdedding in vector db
from google.colab import userdata
import weaviate
import os
import weaviate.classes.config as wvc
from weaviate.auth import AuthApiKey
# from weaviate.util import generate_uuid5
weaviate_url = userdata.get("WEAVIATE_URL")
print(weaviate_url)

weaviate_api_key = userdata.get("WEAVIATE_API_KEY")
print(weaviate_api_key)
if len(chunks) != len(embeddings):
  raise Exception("The number of chunks and embeddings must be the same")
# connect to wevaite
with weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=AuthApiKey(weaviate_api_key),
) as client:

  dataResult=client.collections.create(
      name="MagicLibrary",
      vector_config=None,
      properties=[
        wvc.Property(name="content", data_type=wvc.DataType.TEXT),
        wvc.Property(name="source", data_type=wvc.DataType.TEXT),
    ]
  )
  # now store the data in weviate
  with dataResult.batch.dynamic() as batch:
    for i,chunk in enumerate(chunks):
      # create the propretires
      properties={
          "content":chunk,
          "source":f"source_{i}"
      }
      # send the data the vector dv
      batch.add_object(
          properties,
         vector=embeddings[i]
      )



In [ ]:
# make semlarity search
with weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=weaviate_api_key,
) as client:
  library=client.collections.get("MagicLibrary")
  userInput="How do I bake a chocolate cake?"
  # embend user input
  userInputEmbended=model.encode(userInput).tolist()
  # make the query
  response=library.query.near_vector(
      near_vector=userInputEmbended,
      limit=2
  )
  for obj in response.objects:
    print(f"Match found: {obj.properties['content']}")

In [ ]:
!pip install nbstripout
# nbstripout  Llmv1.ipynb

In [ ]:
!nbstripout https://colab.research.google.com/drive/1I4aSBOg4NO0bhLB_AEHfqPD42CYo6xtE?usp=drive_link